# GitHub Flow와 PR 리뷰

모든 저장소는 임시 폴더에 만들며 실제 GitHub 계정이나 원격 저장소를 변경하지 않습니다.

## 명령을 실행하기 전에 알아둘 Git 상태

Git은 파일을 세 자리에서 봅니다. **작업 폴더(working tree)** 는 지금 편집하는 파일이 있는 자리입니다. **색인(index)** 은 다음 커밋에 넣을 파일 상태를 골라 두는 자리이며, 스테이징 영역(staging area)이라고도 합니다. **커밋(commit)** 은 색인에 고른 상태를 저장소 이력에 기록한 결과입니다. 따라서 `git add app.py`는 파일을 영구 저장하는 명령이 아니라 현재 `app.py`를 색인에 올리는 명령입니다. 이어서 `git commit -m "메시지"`를 실행해야 그 상태가 커밋으로 기록됩니다.

**HEAD** 는 현재 체크아웃한 커밋을 가리킵니다. 보통 HEAD는 `main` 같은 **브랜치(branch)** 이름을 가리키고, 브랜치 이름은 다시 최신 커밋을 가리킵니다. 새 커밋을 만들면 현재 브랜치가 새 커밋으로 이동합니다. 이 관계를 알고 나면 `HEAD~1`은 “현재 커밋의 첫 번째 부모 한 단계 전”, `origin/main`은 “마지막으로 확인한 원격 `main`의 위치”로 읽을 수 있습니다.

명령은 왼쪽부터 `git` → 하위 명령 → 옵션 → 대상으로 읽습니다. 예를 들어 `git diff --cached -- app.py`에서 `diff`는 차이를 보고, `--cached`는 색인과 현재 커밋을 비교하며, `--`는 뒤의 `app.py`를 옵션이 아닌 파일 경로로 구분합니다. 코드의 `git("diff", "--cached", "--", "app.py", cwd=repo)`는 터미널에서 같은 명령을 실행하도록 인자를 나눈 Python 표현입니다.


## 준비

로컬 bare 저장소를 원격 저장소처럼 사용합니다. **원격(remote)** 은 네트워크 자체가 아니라 다른 Git 저장소의 위치에 붙인 이름입니다. `git clone 원격경로 author`는 원격의 이력과 브랜치를 받아 `author` 작업 폴더를 만들고, 그 원격을 관례적인 이름인 `origin`으로 등록합니다. **bare 저장소**에는 편집용 작업 폴더가 없어서 중앙 공유 저장소 역할에 알맞습니다.

`git push -u origin main`은 현재 `main` 커밋을 원격 `main`에 보내며, `-u`는 이후 `git push`가 사용할 추적 관계를 기록합니다. push는 작업 폴더의 미커밋 파일을 보내지 않고 커밋과 참조를 전송합니다.

In [1]:
# 셀이 만든 저장소만 사용하는 임시 작업 폴더를 준비합니다.
import subprocess
import sys
import tempfile
from pathlib import Path
workspace = tempfile.TemporaryDirectory(prefix="git-lesson-")
root = Path(workspace.name)
# cwd는 명령을 실행할 폴더이며 check=True는 명령 실패를 예외로 알립니다.
def git(*args: str, cwd: Path | None = None, check: bool = True) -> str:
    """지정한 임시 저장소에서 Git 명령을 실행합니다."""
    result = subprocess.run(["git", *args], cwd=cwd or root,
                            text=True, capture_output=True, check=check)
    return result.stdout.strip() or result.stderr.strip()
def configure(repo: Path) -> None:
    """실습 저장소에만 작성자 정보를 설정합니다."""
    git("config", "user.name", "Git Learner", cwd=repo)
    git("config", "user.email", "learner@example.invalid", cwd=repo)

In [2]:
# bare 저장소는 원격 역할만 담당합니다.
remote = root / "team.git"
git("init", "--bare", "--initial-branch=main", remote.as_posix())
author = root / "author"
git("clone", remote.as_posix(), author.as_posix())
configure(author)
(author / "README.md").write_text("# 주문 서비스\n", encoding="utf-8")
template = author / ".github" / "pull_request_template.md"
template.parent.mkdir()
template.write_text("## 변경\n- 변경 이유\n\n## 확인\n- [ ] 로컬 실행\n", encoding="utf-8")
git("add", "README.md", template.relative_to(author).as_posix(), cwd=author)
git("commit", "-m", "docs: 저장소와 PR 템플릿 시작", cwd=author)
git("push", "-u", "origin", "main", cwd=author)
print(git("remote", "-v", cwd=author).splitlines()[0])

origin	/var/folders/14/3sq03f6s3_7bs0tygfqqvc_c0000gn/T/git-lesson-vm9bimbk/team.git (fetch)


## 기본 실행

기능 브랜치를 만들고 커밋한 뒤 원격에 게시합니다. `git switch -c feature/order-summary`에서 `switch`는 체크아웃할 브랜치를 바꾸고 `-c`는 현재 커밋에서 새 브랜치를 만듭니다. 이 직후 `main`과 기능 브랜치는 같은 커밋을 가리키지만, 기능 커밋을 만들면 기능 브랜치만 앞으로 이동합니다.

`git branch -vv` 출력에서 별표는 현재 브랜치, 대괄호의 `origin/feature/order-summary`는 추적하는 원격 브랜치입니다. `git diff --stat origin/main...HEAD`의 점 세 개는 두 참조의 공통 조상부터 HEAD까지 기능 브랜치가 추가한 변경을 요약합니다. PR은 이 차이와 변경 이유를 사람에게 검토해 달라고 요청하는 GitHub의 협업 단위입니다.

In [3]:
# 타입힌트는 실행 시 값을 검사하지 않으므로 별도 입력 검증이 필요합니다.
git("switch", "-c", "feature/order-summary", cwd=author)
(author / "summary.py").write_text(
    """def summarize(count: int) -> str:
    return f'주문 {count}건'
""", encoding="utf-8"
)
git("add", "summary.py", cwd=author)
git("commit", "-m", "feat: 주문 요약 함수 추가", cwd=author)
git("push", "-u", "origin", "feature/order-summary", cwd=author)
print(git("branch", "-vv", cwd=author))

* feature/order-summary 73f549a [origin/feature/order-summary] feat: 주문 요약 함수 추가
  main                  c964bc1 [origin/main] docs: 저장소와 PR 템플릿 시작


## PR 자료와 변경 범위

기본 브랜치에 준비된 템플릿을 읽고 기능 코드 차이만 확인합니다. `--stat`은 실제 코드 대신 바뀐 파일과 줄 수를 요약합니다. 리뷰할 때는 요약으로 범위를 확인한 뒤 실제 diff도 읽어야 합니다.

GitHub Flow의 합치기 단계에는 merge, squash merge, rebase merge 같은 선택이 있을 수 있습니다. 이 실습은 부모가 둘인 병합 커밋을 눈으로 확인하기 위해 `git merge --no-ff`를 사용합니다. **rebase**는 한 브랜치의 커밋을 다른 기준 위에서 다시 만든다는 뜻이라 커밋 해시가 바뀝니다. 아직 공유 협업의 기본 동작을 배우는 단계이므로 실습에서는 rebase로 공개 이력을 바꾸지 않습니다.

In [4]:
# 기본 브랜치의 템플릿을 채우고 세 점 비교로 기능 브랜치의 변경을 확인합니다.
pr_body = template.read_text(encoding="utf-8").replace(
    "- 변경 이유", "- 주문 건수 요약 함수 추가"
).replace("- [ ] 로컬 실행", "- [x] 로컬 실행")
print(git("diff", "--stat", "origin/main...HEAD", cwd=author))
print(pr_body)

summary.py | 2 ++
 1 file changed, 2 insertions(+)
## 변경
- 주문 건수 요약 함수 추가

## 확인
- [x] 로컬 실행



## 실패 재현

리뷰어가 빈 문자열 입력에 대한 설명과 처리가 없음을 실제 diff에서 발견합니다.

In [5]:
# 리뷰어의 복제본에서 빈 문자열을 직접 전달해 잘못된 출력을 재현합니다.
reviewer = root / "reviewer"
git("clone", remote.as_posix(), reviewer.as_posix())
configure(reviewer)
git("switch", "feature/order-summary", cwd=reviewer)
before = (reviewer / "summary.py").read_text(encoding="utf-8")
print(before)
print("리뷰 의견: count가 빈 문자열이면 주문 건수가 비어 보입니다.")
print("재현:", subprocess.check_output(
    [sys.executable, "-c", "from summary import summarize; print(summarize(''))"],
    cwd=reviewer, text=True,
).strip())

def summarize(count: int) -> str:
    return f'주문 {count}건'

리뷰 의견: count가 빈 문자열이면 주문 건수가 비어 보입니다.
재현: 주문 건


## 수정

작성자가 입력을 정수로 제한하고 리뷰어가 원격 변경을 받아 병합합니다.

In [6]:
# 수정 커밋을 공유하고 실제 TypeError와 원격 main 갱신을 함께 확인합니다.
(author / "summary.py").write_text(
    """def summarize(count: int) -> str:
    if not isinstance(count, int):
        raise TypeError('count는 정수여야 합니다')
    return f'주문 {count}건'
""", encoding="utf-8"
)
git("add", "summary.py", cwd=author)
git("commit", "-m", "fix: 주문 건수 입력 확인", cwd=author)
git("push", cwd=author)
git("fetch", "origin", cwd=reviewer)
git("switch", "main", cwd=reviewer)
git("merge", "--no-ff", "origin/feature/order-summary", "-m", "merge: 주문 요약", cwd=reviewer)
git("push", "origin", "main", cwd=reviewer)
print(git("log", "--oneline", "--graph", "-4", cwd=reviewer))
check = subprocess.run([sys.executable, "-c",
    "from summary import summarize; summarize('')"], cwd=reviewer, text=True, capture_output=True)
print("빈 문자열 종료 코드:", check.returncode)
print("입력 오류:", check.stderr.splitlines()[-1])
assert check.returncode != 0 and "TypeError: count는 정수여야 합니다" in check.stderr
assert git("rev-parse", "HEAD", cwd=reviewer) == git("rev-parse", "origin/main", cwd=reviewer)

*   da27246 merge: 주문 요약
|\  
| * 0707207 fix: 주문 건수 입력 확인
| * 73f549a feat: 주문 요약 함수 추가
|/  
* c964bc1 docs: 저장소와 PR 템플릿 시작
빈 문자열 종료 코드: 1
입력 오류: TypeError: count는 정수여야 합니다
